Quick sanity check: how are different models currently embedding certain users? \
(big matrix only, to prevent leakage)

In [1]:
# Load in big matrix

import deep_linear_bandits.data as dlb_data

DLB_DIR = "/home/sulay/deep-linear-bandits/"

big_train, _ = dlb_data.preprocess_krbig_interactions(data_dir=DLB_DIR+"kuairec/data/")

In [ ]:
# Get item popularities in big matrix

import numpy as np
import pandas as pd

item_popularity = dlb_data.compute_item_popularity(data_dir=DLB_DIR+"kuairec/data/")

array([13.,  1., 13., ...,  1.,  4.,  2.], shape=(10728,))

In [44]:
# Show items sorted by global popularity

pd.DataFrame({
    'global_popularity': item_popularity
}).sort_values(by=['global_popularity'], ascending=False)

,global_popularity
314,2559.0
7383,2332.0
8298,2253.0
8366,1967.0
3400,1829.0
5464,1802.0
2123,1582.0
2894,1579.0
6787,1500.0
211,1485.0


In [71]:
# Get all user & item embeddings for a specific MODEL

import deep_linear_bandits.two_tower as dlb_tt
import torch
import pickle

pd.set_option('display.min_rows', 25)

NUM_USERS = 7176
NUM_ITEMS = 10728

MODEL = "lr-5em4"

torch.set_float32_matmul_precision('high')
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device('cpu')
)

with open(DLB_DIR + f"tt-models/{MODEL}/model_args.pkl", "rb") as f:
    model_args = pickle.load(f)
model = dlb_tt.TwoTower(**model_args).to(device)
model.load_state_dict(
    torch.load(
        DLB_DIR + f"tt-models/{MODEL}/model.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()

with torch.inference_mode():
    """
    Retrieve embeddings for all users & items in the big matrix
    """
    user_ids = torch.arange(0, NUM_USERS, device=device, dtype=torch.long)
    (user_cat_feats, _), user_numeric_feats = dlb_data.preprocess_user_features(data_dir=DLB_DIR+"kuairec/data/")
    user_cat_feats = torch.tensor(user_cat_feats, dtype=torch.long, device=device)
    user_numeric_feats = torch.tensor(user_numeric_feats, dtype=torch.float32, device=device)

    user_embeddings = model.user_tower(user_ids, user_cat_feats, user_numeric_feats).cpu().numpy()

    item_ids = torch.arange(0, NUM_ITEMS, device=device, dtype=torch.long)
    item_categories = dlb_data.preprocess_item_categories(data_dir=DLB_DIR+"kuairec/data/").to(device)

    item_embeddings = model.item_tower(item_ids, item_categories).cpu().numpy()

In [72]:
# What's a specific user U's top items in the big matrix?
# (from training, avoid leakage of val)

U = 1376 # (recall users are between 0 and 7175)

U_wrs = big_train[big_train.user_id == U].sort_values(by='watch_ratio', ascending=False)
U_wrs = U_wrs.rename(columns = {'video_id': 'item_id'})

# What's the model currently got as the top items for that user?
# user_embeddings: (7176, D); for one user it's (D,)
# item_embeddings: (10728, D)
item_scores = item_embeddings @ user_embeddings[U, :]
top_items = np.argsort(item_scores)[::-1]
top_items_scores = item_scores[top_items]

df = pd.DataFrame({
    'item_id': top_items,
    'model_similarity': top_items_scores
})

# Top items by model similarity, and the actual watch_ratio (+ item popularity)
top_model = df.merge(right=U_wrs, on='item_id')
top_model['global_popularity'] = item_popularity[top_model.item_id.values]
top_model

,item_id,model_similarity,user_id,watch_ratio,global_popularity
0,314,0.762838,1376,2.399087,2559.0
1,5464,0.760125,1376,1.774618,1802.0
2,4040,0.743471,1376,2.753567,1481.0
3,8298,0.733474,1376,2.102752,2253.0
4,4123,0.728501,1376,1.582484,1455.0
5,3400,0.727410,1376,2.256117,1829.0
6,2123,0.721359,1376,1.610614,1582.0
7,154,0.718464,1376,2.291443,1322.0
8,2894,0.716359,1376,1.399328,1579.0
9,5434,0.712953,1376,2.527463,1193.0


In [73]:
# Top items by their actual watch_ratio; index will be where the model is actually placing them
top_model.sort_values(by=['watch_ratio'], ascending=False)

,item_id,model_similarity,user_id,watch_ratio,global_popularity
512,10184,0.406994,1376,7.392721,430.0
26,6463,0.661792,1376,6.188506,464.0
24,3401,0.664566,1376,4.419501,617.0
1486,867,0.328030,1376,3.951360,115.0
1194,5738,0.366673,1376,3.612493,97.0
522,2110,0.406189,1376,3.318202,16.0
1453,7296,0.338547,1376,3.276265,51.0
1185,4261,0.367285,1376,3.173942,162.0
660,4425,0.396986,1376,3.097894,140.0
10,1894,0.706937,1376,2.986958,658.0


In [74]:
# Top items by their global popularity, index is where the model is placing them
top_model.sort_values(by=['global_popularity'], ascending=False)

,item_id,model_similarity,user_id,watch_ratio,global_popularity
0,314,0.762838,1376,2.399087,2559.0
3,8298,0.733474,1376,2.102752,2253.0
5,3400,0.727410,1376,2.256117,1829.0
1,5464,0.760125,1376,1.774618,1802.0
6,2123,0.721359,1376,1.610614,1582.0
8,2894,0.716359,1376,1.399328,1579.0
2,4040,0.743471,1376,2.753567,1481.0
4,4123,0.728501,1376,1.582484,1455.0
25,1709,0.663873,1376,2.021032,1434.0
13,2130,0.697039,1376,2.723297,1417.0
